# CISC3024 Pattern Recognition, AI Assignment #3
## Ablations

Three questions, one pass over the data each:

1. **Backbone** — is DINOv2-G worth 1.1 B parameters against DINOv2-S and -L
   at matched relative depth?
2. **Retained variance** — is `pca_ev=0.99` a reasonable operating point?
3. **Image aggregation** — does the top-1% mean actually beat max and mean?

The main table is the baseline for all three; run notebook 01 first.

### Why this notebook does not just call `main.py` five times

The forward pass is the whole cost. After it, an eigendecomposition, a
projection and a top-k average cost seconds. `main.py` welds the two together,
so sweeping five values means five full runs. This notebook extracts features
once per backbone, keeps the untruncated eigendecomposition, and then evaluates
all twelve `(pca_ev, img_score_agg)` combinations against the same features.
Same numbers, roughly a tenth of the GPU time.

Nothing numerical is reimplemented — the PCA, the scoring, the post-processing
and AU-PRO all come from the repository's own functions.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration. Change these only if you know why.
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/CLendering/SubspaceAD.git"
COMMIT   = "ef56d5c8ab2f1feb7dda1c93b25cc3f73f0960d7"   # 2026-08-17
WORKDIR  = "/content/subspacead_work"
REPO     = f"{WORKDIR}/SubspaceAD"
DS_ROOT  = f"{WORKDIR}/datasets/mvtec-ad"

# Everything that must survive the VM goes here. See the mount cell.
DRIVE_DIR   = "/content/drive/MyDrive/CISC3024_HW3"
RESULT_ROOT = f"{DRIVE_DIR}/results"
LOG_ROOT    = f"{DRIVE_DIR}/logs"

RES      = 672
BACKBONE = "facebook/dinov2-with-registers-giant"
LAYERS   = "-12,-13,-14,-15,-16,-17,-18"
BATCH    = 1
SEED     = 42
KS       = [1, 2, 4]

# Order in which the k values are run. Reorder it when the session is short:
# the loop stops when the quota does, so whatever sits last may not run. The
# order changes nothing about the numbers, only which one survives a cut.
K_ORDER  = KS

# fp16 or fp32. The released code loads fp32; fp16 is a deliberate deviation
# that makes a T4 run feasible. The smoke cell measures the resulting drift on
# a category that was already run in fp32, so the deviation is quantified
# rather than assumed.
PRECISION = "fp16"

CATEGORIES = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut",
              "leather", "metal_nut", "pill", "screw", "tile", "toothbrush",
              "transistor", "wood", "zipper"]

import os, sys, shutil, subprocess, pathlib, textwrap
pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)
print("workdir :", WORKDIR)
print("python  :", sys.version.split()[0])

In [ ]:
import torch, shutil, platform

print("platform :", platform.platform())
print("torch    :", torch.__version__)
print("cuda     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu      :", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"vram     : {total/2**30:.1f} GB total, {free/2**30:.1f} GB free")
    print("capability:", torch.cuda.get_device_capability(0))
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU.")

du = shutil.disk_usage("/content")
print(f"disk     : {du.free/2**30:.1f} GB free of {du.total/2**30:.1f} GB")

# A DINOv2-G checkpoint plus the dataset plus features do not fit in much less.
assert du.free > 25 * 2**30, "Need at least 25 GB free disk."

In [ ]:
# ---------------------------------------------------------------------------
# Results go to Drive, not to the VM.
#
# A free Colab runtime is recycled without warning and /content goes with it.
# The first attempt at this experiment did exactly that: it reached k=4, ran
# out of GPU quota mid-category, and lost two complete k values plus four
# categories of the third, because the only copy of every CSV was on the
# ephemeral disk.
#
# With results on Drive the unit of loss becomes one category, about seven
# minutes, and a session can be stopped at any point without cost. That is
# what makes an exhausted quota an inconvenience instead of a restart.
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

for d in (RESULT_ROOT, LOG_ROOT):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

free = shutil.disk_usage("/content/drive/MyDrive").free / 2**30
print("result root:", RESULT_ROOT)
print(f"drive free : {free:.1f} GB")
assert free > 1, "Drive is nearly full; the results need a few hundred MB."

# A short write probe. Drive is a network filesystem and can be read-only when
# the mount is half-up; better to find out here than after a category finishes.
probe = pathlib.Path(RESULT_ROOT) / ".write_probe"
probe.write_text("ok")
assert probe.read_text() == "ok"
probe.unlink()
print("drive is writable")

In [ ]:
if os.path.isdir(REPO):
    print("repo already present")
else:
    print(subprocess.run(["git", "clone", REPO_URL, REPO],
                         capture_output=True, text=True).stdout)

subprocess.run(["git", "-C", REPO, "checkout", "--quiet", COMMIT], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
assert head == COMMIT, f"Expected {COMMIT}, got {head}. The code below was written against that commit."

In [ ]:
# Only what is actually imported. anomalib is listed in the repo's
# requirements.txt but no module imports it, and installing it drags in
# PyTorch Lightning, which can replace Colab's pinned torch build.
# scipy is imported by main.py but missing from requirements.txt; Colab ships it.
import importlib, subprocess, sys

for mod in ["scipy", "sklearn", "cv2", "pandas", "tqdm", "transformers", "kornia"]:
    try:
        importlib.import_module(mod)
        print(f"{mod:14s} present")
    except ImportError:
        print(f"{mod:14s} MISSING -> installing")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)

# Install the package without its declared dependencies, so anomalib is skipped.
# --no-deps still puts src/ on sys.path, which the absolute imports need.
print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"],
                     capture_output=True, text=True).stdout)

import torch
print("torch after install:", torch.__version__, "| cuda build:", torch.version.cuda)

In [ ]:
# ---------------------------------------------------------------------------
# Patch 1: do not request attention maps unless they are used.
#
# extractor.py calls the backbone with output_attentions=True unconditionally.
# At 672 px a ViT-g/14 sees 2,309 tokens; keeping the attention tensor of all
# 40 layers in FP32 is 40 * 16 * 2309**2 * 4 bytes = 13.6 GB, on top of 4.5 GB
# of weights. That does not fit on a T4. Attention maps are only consumed by
# --bg_mask_method dino_saliency, which this study does not use.
# ---------------------------------------------------------------------------
import pathlib

EXTRACTOR = pathlib.Path(REPO) / "src/subspacead/core/extractor.py"
src = EXTRACTOR.read_text()

if "_NEED_ATTN" in src:
    print("shim already applied")
else:
    anchor = 'DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")'
    assert src.count(anchor) == 1, "anchor not found; extractor.py changed"
    src = src.replace(anchor, anchor + textwrap.dedent("""
    
        # SHIM: attention maps are only needed for --bg_mask_method dino_saliency.
        _NEED_ATTN = False"""), 1)

    assert src.count("output_attentions=True") == 1
    src = src.replace("output_attentions=True", "output_attentions=_NEED_ATTN", 1)

    assert src.count("if attentions is None:") == 1
    src = src.replace("if attentions is None:", "if attentions is None and _NEED_ATTN:", 1)

    doc = '"""Extracts the DINO saliency mask from attention weights."""'
    assert src.count(doc) == 1
    src = src.replace(doc, doc + "\n        if not _NEED_ATTN:\n            return None", 1)

    EXTRACTOR.write_text(src)
    print("shim applied")

print("-" * 70)
print(subprocess.run(["grep", "-n", "_NEED_ATTN", str(EXTRACTOR)],
                     capture_output=True, text=True).stdout)

# Import check: the package must load with the patch in place.
sys.path.insert(0, REPO)
sys.path.insert(0, f"{REPO}/src")
import src.subspacead.core.extractor as ex
print("import ok, _NEED_ATTN =", ex._NEED_ATTN)
assert ex._NEED_ATTN is False

In [ ]:
# ---------------------------------------------------------------------------
# Patch 2: numpy 2.x removed np.trapz.
#
# main.py line 188 computes AU-PRO with np.trapz, which no longer exists in
# numpy >= 2.0 (it was renamed np.trapezoid). The repo does not pin numpy, so
# on a current Colab the run dies while computing the first category's AU-PRO,
# after the features have already been extracted. The rename is exact: same
# function, same result.
# ---------------------------------------------------------------------------
import numpy as np

MAIN_PY = pathlib.Path(REPO) / "main.py"
msrc = MAIN_PY.read_text()

if hasattr(np, "trapz"):
    print(f"numpy {np.__version__} still has np.trapz, no patch needed")
elif "np.trapz(" in msrc:
    MAIN_PY.write_text(msrc.replace("np.trapz(", "np.trapezoid("))
    print(f"numpy {np.__version__}: patched np.trapz -> np.trapezoid in main.py")
else:
    print("main.py already patched")

print(subprocess.run(["grep", "-n", "trapezoid", str(MAIN_PY)],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---------------------------------------------------------------------------
# Patch 3: stop boxing every pixel into a Python object.
#
# main.py collects per-pixel scores with
#     px_pred_all_auroc.extend(anomaly_map.flatten().astype(np.float32))
# extend() on a numpy array pushes one Python scalar per element into the list.
# At 672 px an image has 451,584 pixels, a boxed float costs 24 bytes plus an
# 8-byte pointer, so each of the two score lists grows by ~2.2 GB per 150
# images, and the uint8 label list by ~2.4 GB. MVTec-AD's transistor split has
# 240 test images, which needs ~11 GB of list alone; the kernel is killed with
# exit 137 before the metrics are printed, and because the CSV is only written
# after the last category, every finished category is lost with it.
#
# The replacement stores the per-image arrays and concatenates once, which is
# 6x smaller. The values are unchanged: float32 -> Python float -> float64 is an
# exact embedding, so casting the concatenation to float64 reproduces exactly
# what np.array(list_of_floats) produced.
# ---------------------------------------------------------------------------
import numpy as np, pathlib

MAIN_PY = pathlib.Path(REPO) / "main.py"
src = MAIN_PY.read_text()

if "_Chunks" in src:
    print("memory patch already applied")
else:
    helper = """
class _Chunks:
    # Holds per-image arrays and concatenates them once at the end.
    # Stands in for list.extend(arr) on a per-element basis, which boxes every
    # element into a Python object and costs 4-8x the memory of the array.

    __slots__ = ("chunks",)

    def __init__(self):
        self.chunks = []

    def extend(self, arr):
        self.chunks.append(np.asarray(arr).ravel())

    def array(self, dtype=None):
        if not self.chunks:
            return np.array([], dtype=dtype)
        out = np.concatenate(self.chunks)
        return out.astype(dtype, copy=False) if dtype is not None else out


"""
    anchor = "def compute_aupro("
    assert src.count(anchor) == 1, "anchor not found; main.py changed"
    src = src.replace(anchor, helper.lstrip("\n") + anchor, 1)

    pairs = [
        ("        px_true_all = []\n",
         "        px_true_all = _Chunks()\n"),
        ("        px_pred_all_auroc = []\n",
         "        px_pred_all_auroc = _Chunks()\n"),
        ("        px_pred_all_normalized = []\n",
         "        px_pred_all_normalized = _Chunks()\n"),
        ("        px_true_arr = np.array(px_true_all, dtype=np.uint8)\n",
         "        px_true_arr = px_true_all.array(np.uint8)\n"),
        ("        px_pred_arr_auroc = np.array(px_pred_all_auroc)\n",
         "        px_pred_arr_auroc = px_pred_all_auroc.array(np.float64)\n"),
        ("        px_pred_arr_normalized = np.array(px_pred_all_normalized)\n",
         "        px_pred_arr_normalized = px_pred_all_normalized.array(np.float64)\n"),
    ]
    for old, new in pairs:
        assert src.count(old) == 1, f"expected exactly one occurrence of {old!r}"
        src = src.replace(old, new, 1)

    MAIN_PY.write_text(src)
    print("memory patch applied")

print("-" * 70)
print(subprocess.run(["grep", "-n", "_Chunks", str(MAIN_PY)],
                     capture_output=True, text=True).stdout)

# The substitution has to be exact, not merely close. Reproduce both paths on
# the same data and compare bit for bit.
import numpy as np
rng = np.random.default_rng(0)
for dt in (np.uint8, np.float32):
    chunks = [(rng.random(997) * 255).astype(dt) for _ in range(7)]
    old = np.array([v for c in chunks for v in c])          # list.extend path
    new = np.concatenate(chunks).astype(
        np.float64 if dt == np.float32 else np.uint8)       # _Chunks path
    assert np.array_equal(old, new), f"mismatch for {dt}"
    print(f"{np.dtype(dt).name:8s} identical over {old.size} values, dtype {old.dtype}")

In [ ]:
# ---------------------------------------------------------------------------
# Patch 4: load the backbone in half precision.
#
# The released code calls AutoModel.from_pretrained(model_ckpt) with no dtype,
# so the weights arrive in fp32. On a T4 that is 4.5 GB of weights and a
# measured 3.32 s per image at 672 px, which puts the three main runs at about
# 11 hours. fp16 weights are 2.25 GB and the forward pass uses the tensor cores.
#
# Three changes, nothing else:
#   1. load the weights in fp16,
#   2. cast pixel_values to fp16 so the first matmul does not upcast,
#   3. upcast the aggregated patch features back to fp32 before numpy, so the
#      streaming PCA accumulates in float64 from fp32 inputs rather than fp16.
#
# LayerNorm and softmax keep fp32 accumulators inside ATen on CUDA, so this
# does not need autocast to stay numerically stable. The drift is measured in
# the smoke cell against a category already run in fp32, not assumed.
# ---------------------------------------------------------------------------
import pathlib, subprocess

EXTRACTOR = pathlib.Path(REPO) / "src/subspacead/core/extractor.py"
src = EXTRACTOR.read_text()

if "_FP16" in src:
    print("precision patch already applied")
else:
    anchor = "class FeatureExtractor:"
    assert src.count(anchor) == 1
    src = src.replace(
        anchor,
        f"_FP16 = {PRECISION == 'fp16'}   # set by the notebook\n\n\n" + anchor,
        1,
    )

    old = "        self.model = AutoModel.from_pretrained(model_ckpt).eval().to(DEVICE)"
    assert src.count(old) == 1, "from_pretrained line changed"
    src = src.replace(
        old,
        "        self.model = AutoModel.from_pretrained(\n"
        "            model_ckpt, torch_dtype=torch.float16 if _FP16 else None\n"
        "        ).eval().to(DEVICE)",
        1,
    )

    old = "        ).to(DEVICE)\n"
    assert src.count(old) == 1, "processor call changed"
    src = src.replace(
        old,
        "        )\n"
        "        if _FP16:\n"
        '            inputs["pixel_values"] = inputs["pixel_values"].half()\n'
        "        inputs = inputs.to(DEVICE)\n",
        1,
    )

    old = "        return fused.cpu().numpy()\n"
    assert src.count(old) == 1, "feature return changed"
    src = src.replace(old, "        return fused.cpu().float().numpy()\n", 1)

    EXTRACTOR.write_text(src)
    print(f"precision patch applied, _FP16 = {PRECISION == 'fp16'}")

print("-" * 70)
print(subprocess.run(["grep", "-n", "_FP16\\|torch.float16\\|cpu().float()",
                      str(EXTRACTOR)], capture_output=True, text=True).stdout)

## Data

Same archive as notebook 01. If this session already extracted it, this cell
does nothing; otherwise it downloads and unpacks.

In [ ]:
# ---------------------------------------------------------------------------
# MVTec-AD. The official site now serves the archive through a browser session
# (a direct request returns an 11 KB redirect page), so the download goes
# through the Hugging Face mirror that hosts the original archive.
# If that fails, upload the archive yourself and set MANUAL_ARCHIVE.
# ---------------------------------------------------------------------------
import os, shutil, zipfile, pathlib, subprocess

MANUAL_ARCHIVE = None   # e.g. "/content/mvtec_anomaly_detection.zip"

ds = pathlib.Path(DS_ROOT)
already = ds.is_dir() and all((ds / c).is_dir() for c in CATEGORIES)

if already:
    print("dataset already extracted")
else:
    if MANUAL_ARCHIVE is None:
        from huggingface_hub import hf_hub_download
        archive = hf_hub_download(
            repo_id="hdtech/mvtech_anomaly_detection",
            filename="mvtech_anomaly_detection.zip",
            repo_type="dataset",
            local_dir=f"{WORKDIR}/_dl",
        )
    else:
        archive = MANUAL_ARCHIVE
    print("archive:", archive, f"{os.path.getsize(archive)/2**30:.2f} GB")

    tmp = pathlib.Path(f"{WORKDIR}/_extract")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    print("extracting ...")
    shutil.unpack_archive(archive, tmp)

    # The archive nests the categories at an unknown depth; find the shallowest
    # directory that holds all fifteen.
    found = None
    for root, dirs, files in os.walk(tmp):
        if all((pathlib.Path(root) / c).is_dir() for c in CATEGORIES):
            found = pathlib.Path(root)
            break
    assert found is not None, "no directory containing all 15 categories"
    print("category root:", found)

    shutil.rmtree(ds, ignore_errors=True)
    ds.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(found), str(ds))
    shutil.rmtree(tmp, ignore_errors=True)
    print("moved to", ds)

In [ ]:
# ---------------------------------------------------------------------------
# What the ablation runs on.
#
# Six categories rather than fifteen. The ablation is about the shape of the
# curve, not the last decimal of the mean, and the six chosen here span the
# failure modes the main table already exposed:
#
#   bottle      easy, rigid object, saturated at 1.000
#   carpet      easy, texture, saturated at 1.000
#   capsule     the one category that collapsed at k=1 (0.6745)
#   cable       thin structure, lowest non-collapsed image AUROC (0.9196)
#   screw       small object, second lowest (0.8994)
#   transistor  no augmentation upstream, and AU-PRO 0.699 against ~0.95 elsewhere
#
# Everything is otherwise held at the main run's settings, so the giant arm at
# pca_ev=0.99 with mtop1p has to land on the main table's numbers for these six
# categories. That check is the last cell of this notebook and it is the only
# reason to trust the rest.
# ---------------------------------------------------------------------------
ABL_CATS = ["bottle", "carpet", "capsule", "cable", "screw", "transistor"]

K_SHOT     = 4            # matches the main table's largest k
AUG_COUNT  = 30
AUG_LIST   = ["rotate"]   # upstream default; the main run did not override it
NO_AUG     = ["transistor"]   # upstream default in config.py
RES        = 672
SEED       = 42

EVS  = [0.95, 0.99, 1.0]
AGGS = ["mtop1p", "max", "mean", "p99"]

# Batch 1 on the training pass, exactly as the main run. Batch 4 on the test
# pass: no augmentation is drawn there, so the only difference is matmul
# tiling. If the anchor check at the end drifts by more than about 0.005, set
# this to 1 and rerun -- that is the one number in this notebook that trades
# fidelity for time.
BATCH_TRAIN = 1
BATCH_TEST  = 4

# Relative-depth layer strings, as in scripts/backbone_ablation.sh, so the
# compared layers sit at the same 55-75% depth in every backbone.
BACKBONES = {
    "small": ("facebook/dinov2-small", "-4,-5"),
    "large": ("facebook/dinov2-large", "-7,-8,-9,-10,-11"),
    "giant": ("facebook/dinov2-with-registers-giant",
              "-12,-13,-14,-15,-16,-17,-18"),
}

# giant first: it is the anchor, and if the session dies early the one arm that
# matters most is already on Drive.
ARMS = ["giant", "large", "small"]

print("arms:", ARMS)
print("categories:", ABL_CATS)
print(f"grid per arm: {len(EVS)} pca_ev x {len(AGGS)} img_score_agg")

## The harness

One `FeatureExtractor` per backbone, one forward pass per image, every
hyper-parameter evaluated on the cached features. Results are appended to Drive
after each category, so an interrupted session loses at most one category.

In [ ]:
# ---------------------------------------------------------------------------
# Single-pass ablation harness.
#
# The expensive part of this method is the ViT forward pass. Per category at
# k=4: four normal images plus thirty rotations each is 124 images, and the
# two-pass PCA fit walks that set twice, so 248 forwards for the fit alone,
# plus every test image. Across six categories that is roughly 2,150 forwards,
# around half an hour on a T4 at 672 px.
#
# Everything after the forward pass is cheap. One eigendecomposition of a
# 1536x1536 covariance, the choice of k from an explained-variance target, a
# projection, a top-k average: seconds.
#
# main.py welds the two together, so sweeping five hyper-parameter values means
# five full runs and five times the forwards. This harness extracts the
# features once per backbone and keeps the *untruncated* eigendecomposition,
# then evaluates every (pca_ev, img_score_agg) pair against those same
# features. Twelve configurations for the price of one.
#
# Nothing numerical is reimplemented here. The fit is PCAModel.fit, the
# component selection is PCAModel._select_k_components, the score is
# calculate_anomaly_scores, the map post-processing is post_process_map, the
# aggregation is aggregate_image_score and AU-PRO is main.compute_aupro. The
# harness only decides what to call, and in which order.
#
# Two deliberate differences from main.py, both recorded in the report:
#
#   1. main.py seeds once before the category loop, so its RNG stream runs
#      across categories and rerunning one category alone gives different
#      numbers. This seeds per category instead, so every row is independently
#      reproducible.
#   2. main.py selects k inside the PCA fit and throws the rest of the spectrum
#      away. This keeps it, which is what makes the ev sweep free.
# ---------------------------------------------------------------------------
import os, sys, json, math, time, datetime, pathlib, random
import numpy as np, pandas as pd, torch
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score

for _p in (f"{REPO}/src", REPO):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from subspacead.core.extractor import FeatureExtractor
from subspacead.core.pca import PCAModel
from subspacead.post_process.scoring import (
    calculate_anomaly_scores, post_process_map, aggregate_image_score,
)
from subspacead.data.datasets import get_dataset_handler
from subspacead.data.transforms import get_augmentation_transform
from subspacead.config import parse_layer_indices
from main import compute_aupro

ABL_ROOT = f"{DRIVE_DIR}/ablation"
MAP_ROOT = f"{ABL_ROOT}/patch_maps"
for _d in (ABL_ROOT, MAP_ROOT):
    pathlib.Path(_d).mkdir(parents=True, exist_ok=True)

IMG_CSV = f"{ABL_ROOT}/per_category_image.csv"
PIX_CSV = f"{ABL_ROOT}/per_category_pixel.csv"


def _extract(extractor, imgs, res, layers, agg):
    """One forward pass. Returns tokens as (n*h*w, C) plus the patch grid."""
    tokens, (h_p, w_p), _ = extractor.extract_tokens(
        imgs, res, layers, agg, [], False, use_clahe=False, dino_saliency_layer=0,
    )
    return tokens.reshape(-1, tokens.shape[-1]), h_p, w_p


def _train_generator(extractor, paths, aug_tf, aug_count, res, layers, bs):
    """Mirror of main.py's feature_generator_full.

    It re-opens and re-augments on every call, and that is deliberate. The
    two-pass PCA fit calls the generator twice, and upstream the two passes see
    *different* random rotations. Caching the features would make the passes
    consistent -- arguably better statistics -- but it is not the released
    behaviour, and the anchor check would then be measuring the difference
    instead of validating the harness. So the two passes are left alone.
    """
    def gen():
        imgs = []
        for p in paths:
            pil = Image.open(p).convert("RGB")
            imgs.append(pil)
            if aug_tf is not None:
                for _ in range(aug_count):
                    imgs.append(aug_tf(pil))
        for i in range(0, len(imgs), bs):
            X, _, _ = _extract(extractor, imgs[i:i + bs], res, layers, "mean")
            yield X
    return gen


def fit_full_pca(extractor, paths, aug_tf, aug_count, res, layers, bs):
    """Fit PCA with no truncation, so every pca_ev can be sliced out of it."""
    probe, h_p, w_p = _extract(
        extractor, [Image.open(paths[0]).convert("RGB")], res, layers, "mean")
    feature_dim = probe.shape[-1]
    n_img = len(paths) * ((1 + aug_count) if aug_tf is not None else 1)
    total_tokens = n_img * h_p * w_p
    num_batches = math.ceil(n_img / bs)

    # k=None and ev=None together mean "keep every component": with both unset,
    # _select_k_components falls through to self.k = evecs.shape[1].
    pca = PCAModel(k=None, ev=None, whiten=False)
    pca.fit(_train_generator(extractor, paths, aug_tf, aug_count, res, layers, bs),
            feature_dim, total_tokens, num_batches)
    return pca, feature_dim, h_p, w_p


def params_at_ev(pca, ev):
    """Re-run the component selection on the cached spectrum, at a new ev."""
    m = PCAModel(k=None, ev=ev, whiten=False)
    m.mu_ = pca.mu_
    m.explained_variance_ = pca.explained_variance_
    m._select_k_components(pca.components_)   # full evecs in, truncated out
    return m._build_pca_params(), int(m.k)


def _gt_masks(handler, paths, res):
    """Ground truth at map resolution, resized the same way main.py does."""
    import torchvision.transforms.functional as TF
    out = []
    for p in paths:
        g = handler.get_ground_truth_path(p)
        if not g or not os.path.exists(g):
            out.append(np.zeros((res, res), np.uint8))
            continue
        m = TF.resize(Image.open(g).convert("L"), (res, res),
                      interpolation=TF.InterpolationMode.NEAREST)
        out.append((np.array(m) > 0).astype(np.uint8))
    return out


def _append(df, path):
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def _done(arm, cat):
    if not os.path.exists(IMG_CSV):
        return False
    d = pd.read_csv(IMG_CSV)
    return not d[(d["backbone"] == arm) & (d["category"] == cat)].empty


def eval_arm(arm, ckpt, layers, cats, force=False):
    """Extract once, score every (ev, agg) pair, write each category as it lands."""
    # extract_tokens wants indices, not the "-12,-13,..." string the CLI takes.
    # main.py does this conversion right after get_args(); skipping it fails
    # deep inside the layer aggregation with a tuple-index error.
    layer_str = layers
    layers = parse_layer_indices(layer_str)
    extractor = FeatureExtractor(ckpt)
    for cat in cats:
        tag = f"{arm}/{cat}"
        if not force and _done(arm, cat):
            print(f"{tag}: already on Drive, skipping")
            continue

        t0 = time.time()
        # Per-category seeding: see the header note.
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        handler = get_dataset_handler("mvtec_ad", DS_ROOT, cat)
        train = handler.get_train_paths()
        test = handler.get_test_paths()

        # K-shot sampling, identical to main.py: shuffle, take the first k.
        random.shuffle(train)
        train = train[:K_SHOT]

        aug_tf = get_augmentation_transform(AUG_LIST, RES) if AUG_COUNT > 0 else None
        if cat in NO_AUG:
            aug_tf = None
        n_fit_imgs = len(train) * ((1 + AUG_COUNT) if aug_tf is not None else 1)

        pca, fdim, h_p, w_p = fit_full_pca(
            extractor, train, aug_tf, AUG_COUNT, RES, layers, BATCH_TRAIN)

        params, ks = {}, {}
        for ev in EVS:
            params[ev], ks[ev] = params_at_ev(pca, ev)

        print(f"{tag}: {len(train)} fit images ({n_fit_imgs} with aug), "
              f"dim {fdim}, grid {h_p}x{w_p}, k = {ks}, "
              f"fit {time.time()-t0:.0f}s", flush=True)

        # One pass over the test set; every ev is scored from the same features.
        n = len(test)
        labels = np.array([0 if "good" in str(p) else 1 for p in test], np.uint8)
        maps = {ev: np.zeros((n, h_p, w_p), np.float32) for ev in EVS}

        for i in range(0, n, BATCH_TEST):
            chunk = test[i:i + BATCH_TEST]
            imgs = [Image.open(p).convert("RGB") for p in chunk]
            X, _, _ = _extract(extractor, imgs, RES, layers, "mean")
            for ev in EVS:
                s = calculate_anomaly_scores(X, params[ev], "reconstruction", 0)
                maps[ev][i:i + len(imgs)] = s.reshape(len(imgs), h_p, w_p)

        # The raw patch maps are the only thing worth keeping: 9 KB an image,
        # so a new aggregation later needs no forward pass at all.
        np.savez_compressed(
            f"{MAP_ROOT}/{arm}__{cat}.npz",
            labels=labels, h_p=h_p, w_p=w_p,
            **{f"ev{ev}": maps[ev] for ev in EVS},
        )

        gts = _gt_masks(handler, test, RES)
        px_true = np.concatenate([g.ravel() for g in gts])

        for ev in EVS:
            big = np.stack([post_process_map(maps[ev][j], RES) for j in range(n)])
            p_auroc = float(roc_auc_score(px_true, big.ravel()))
            aupro = float(compute_aupro(big, gts))

            rows = []
            for agg in AGGS:
                sc = np.array([aggregate_image_score(big[j], agg) for j in range(n)])
                rows.append({
                    "backbone": arm, "category": cat, "k_shot": K_SHOT,
                    "pca_ev": ev, "img_score_agg": agg, "k_components": ks[ev],
                    "Image AUROC": float(roc_auc_score(labels, sc)),
                    "Image AUPR": float(average_precision_score(labels, sc)),
                })
            _append(pd.DataFrame(rows), IMG_CSV)
            _append(pd.DataFrame([{
                "backbone": arm, "category": cat, "k_shot": K_SHOT,
                "pca_ev": ev, "k_components": ks[ev],
                "Pixel AUROC": p_auroc, "AU-PRO": aupro,
            }]), PIX_CSV)
            print(f"  ev={ev:<5} k={ks[ev]:<5} pixel AUROC {p_auroc:.4f}  "
                  f"AU-PRO {aupro:.4f}", flush=True)
            del big

        print(f"{tag}: done in {(time.time()-t0)/60:.1f} min", flush=True)

        meta = {
            "backbone": arm, "model_ckpt": ckpt, "layers": layer_str,
            "categories": ABL_CATS, "k_shot": K_SHOT, "aug_count": AUG_COUNT,
            "aug_list": AUG_LIST, "no_aug": NO_AUG, "image_res": RES,
            "pca_ev": EVS, "img_score_agg": AGGS,
            "batch_train": BATCH_TRAIN, "batch_test": BATCH_TEST,
            "seed": SEED, "precision": PRECISION,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "torch": torch.__version__,
            "updated": datetime.datetime.now().isoformat(timespec="seconds"),
        }
        pathlib.Path(f"{ABL_ROOT}/meta_{arm}.json").write_text(json.dumps(meta, indent=2))

## Run

`giant` first — it is the anchor, and if the session dies early the arm that
matters most is already saved. Each arm re-runs from Drive on a restart and
skips whatever is already there.

In [ ]:
t_start = time.time()
for arm in ARMS:
    ckpt, layers = BACKBONES[arm]
    print("=" * 78)
    print(f"arm {arm}  {ckpt}  layers {layers}")
    eval_arm(arm, ckpt, layers, ABL_CATS)
print(f"\nall arms finished in {(time.time()-t_start)/60:.1f} min")

## Results

In [ ]:
import pandas as pd

img = pd.read_csv(IMG_CSV)
pix = pd.read_csv(PIX_CSV)

g = (img.groupby(["backbone", "pca_ev", "img_score_agg"])
        .agg(n_categories=("category", "nunique"),
             image_auroc=("Image AUROC", "mean"),
             image_aupr=("Image AUPR", "mean"))
        .reset_index())
p = (pix.groupby(["backbone", "pca_ev"])
        .agg(n_categories=("category", "nunique"),
             pixel_auroc=("Pixel AUROC", "mean"),
             aupro=("AU-PRO", "mean"))
        .reset_index())
t = g.merge(p, on=["backbone", "pca_ev", "n_categories"])
t = t.sort_values(["backbone", "pca_ev", "img_score_agg"])
t.to_csv(f"{ABL_ROOT}/summary.csv", index=False)
print(t.to_string(index=False))
print(f"\nsaved {ABL_ROOT}/summary.csv")

## Figures

In [ ]:
# ---------------------------------------------------------------------------
# Three figures, one per sweep. The exact numbers belong in the table; these
# are here to show the shape.
#
# The ev sweep gets two panels. At pca_ev=1.0 the PCA keeps every component,
# the projection becomes the identity, the residual becomes zero and the score
# degrades to float noise: image AUROC falls to 0.38. Drawn on a single axis
# that collapse compresses the 0.95 -> 0.99 difference into nothing, and that
# difference is what the sweep is about. So the left panel shows the collapse
# and the right panel zooms to the working range.
# ---------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

s = pd.read_csv(f"{ABL_ROOT}/summary.csv")
COLS = ["image_auroc", "image_aupr", "pixel_auroc", "aupro"]
NICE = {"image_auroc": "Image AUROC", "image_aupr": "Image AUPR",
        "pixel_auroc": "Pixel AUROC", "aupro": "AU-PRO"}
SHORT = {"image_auroc": "Image\nAUROC", "image_aupr": "Image\nAUPR",
         "pixel_auroc": "Pixel\nAUROC", "aupro": "AU-PRO"}

# --- A. backbone -----------------------------------------------------------
# Metrics on the x-axis, backbones as the series. The other way round needs
# four bars per group and the labels collide, because the four metrics sit
# within 0.06 of each other.
order = ["small", "large", "giant"]
names = {"small": "DINOv2-S (21M)", "large": "DINOv2-L (300M)",
         "giant": "DINOv2-G (1.1B)"}
bb = (s[(s.pca_ev == 0.99) & (s.img_score_agg == "mtop1p")]
      .set_index("backbone").loc[order])

fig, ax = plt.subplots(figsize=(8.2, 3.7))
w = 0.26
for i, b in enumerate(order):
    xs = np.arange(len(COLS)) + (i - 1) * w
    vals = bb.loc[b, COLS].values.astype(float)
    bars = ax.bar(xs, vals, width=w, label=names[b])
    for rect, v in zip(bars, vals):
        ax.text(rect.get_x() + rect.get_width() / 2, v + 0.003, f"{v:.3f}",
                ha="center", va="bottom", fontsize=6.5)
ax.set_xticks(np.arange(len(COLS)))
ax.set_xticklabels([SHORT[c] for c in COLS], fontsize=8)
ax.set_ylim(0.86, 1.0)
ax.set_ylabel("score")
ax.legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.16),
          frameon=False)
ax.set_title("A. Backbone at matched relative depth "
             "(6 categories, k=4, ev=0.99)", fontsize=9)
fig.tight_layout()
fig.savefig(f"{ABL_ROOT}/fig_ablation_backbone.png", dpi=200)
plt.show()

# --- B. retained variance --------------------------------------------------
g = (s[(s.backbone == "giant") & (s.img_score_agg == "mtop1p")]
     .sort_values("pca_ev"))
series = [("image_auroc", "o"), ("pixel_auroc", "s"), ("aupro", "^")]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
for ax, ylim, ttl in [(axes[0], (0.0, 1.05), "full scale"),
                      (axes[1], (0.89, 1.0), "zoomed to the working range")]:
    for c, mk in series:
        ax.plot(g.pca_ev, g[c], marker=mk, label=NICE[c])
    ax.set_xlim(0.94, 1.005)
    ax.set_ylim(*ylim)
    ax.set_xlabel("retained variance (pca_ev)")
    ax.set_title(ttl, fontsize=9)
axes[0].set_ylabel("score")
axes[0].legend(fontsize=7, loc="lower left")
axes[0].annotate("k = D: projection is the\nidentity, residual is zero",
                 xy=(0.997, 0.40), xytext=(0.952, 0.63), fontsize=7,
                 arrowprops=dict(arrowstyle="->", lw=0.8))
fig.suptitle("B. Retained variance, DINOv2-G (6 categories, k=4)", fontsize=9)
fig.tight_layout()
fig.savefig(f"{ABL_ROOT}/fig_ablation_ev.png", dpi=200)
plt.show()

# --- C. aggregation --------------------------------------------------------
order = ["mean", "max", "p99", "mtop1p"]
ag = (s[(s.backbone == "giant") & (s.pca_ev == 0.99)]
      .set_index("img_score_agg").loc[order])

fig, ax = plt.subplots(figsize=(6.5, 3.7))
w = 0.36
for i, c in enumerate(["image_auroc", "image_aupr"]):
    xs = np.arange(len(order)) + (i - 0.5) * w
    bars = ax.bar(xs, ag[c], width=w, label=NICE[c])
    for rect, v in zip(bars, ag[c]):
        ax.text(rect.get_x() + rect.get_width() / 2, v + 0.003, f"{v:.3f}",
                ha="center", va="bottom", fontsize=7)
ax.set_xticks(np.arange(len(order)))
ax.set_xticklabels(order)
ax.set_ylim(0.8, 1.0)
ax.set_ylabel("score")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("C. Image-level aggregation, DINOv2-G "
             "(6 categories, k=4, ev=0.99)", fontsize=9)
fig.tight_layout()
fig.savefig(f"{ABL_ROOT}/fig_ablation_agg.png", dpi=200)
plt.show()

print("figures written to", ABL_ROOT)

## Does the harness agree with the main table?

In [ ]:
# ---------------------------------------------------------------------------
# Correctness gate.
#
# The giant arm at pca_ev=0.99 with mtop1p is the main table's configuration.
# If the harness is faithful, these six categories have to reproduce the main
# table's k=4 rows to within run-to-run noise. We measured that noise at about
# 0.002 on this hardware, so a mean absolute deviation near or below 0.005
# means the harness is doing the same arithmetic as main.py. A deviation in the
# tenths means it is not, and nothing downstream is worth reading.
#
# The reference comes from notebook 01's bundle, which is not necessarily on
# this Drive: 01 was run on a different host and its bundle was downloaded
# rather than left behind. So the file is searched for first, and the values
# below -- the same numbers, copied from that bundle -- are used if it is
# absent. Upload k4_per_category.csv next to this notebook if you would rather
# the check read the file.
# ---------------------------------------------------------------------------
import pathlib
import pandas as pd

REF_K4 = {
    "bottle":     1.0000,
    "carpet":     0.9996,
    "capsule":    0.8620,
    "cable":      0.9196,
    "screw":      0.8994,
    "transistor": 0.9804,
}

ref = None
cands = sorted(pathlib.Path(DRIVE_DIR).rglob("k4_per_category.csv"))
for p in cands:
    try:
        d = pd.read_csv(p)
    except Exception:
        continue
    if {"Category", "Image AUROC"} <= set(d.columns):
        ref = d[d["Category"].isin(ABL_CATS)][["Category", "Image AUROC"]]
        ref.columns = ["category", "main_table"]
        print(f"reference read from {p}")
        break

if ref is None or len(ref) < len(ABL_CATS):
    ref = pd.DataFrame({"category": ABL_CATS,
                        "main_table": [REF_K4[c] for c in ABL_CATS]})
    print("no usable k4_per_category.csv on Drive; using the values copied "
          "from notebook 01's bundle")

if not pathlib.Path(IMG_CSV).exists():
    raise SystemExit("no ablation results yet -- run the Run cell first")

img = pd.read_csv(IMG_CSV)
har = img[(img.backbone == "giant") & ((img.pca_ev - 0.99).abs() < 1e-9)
          & (img.img_score_agg == "mtop1p")][["category", "Image AUROC"]]
har.columns = ["category", "harness"]

missing = sorted(set(ABL_CATS) - set(har["category"]))
if missing:
    print(f"giant arm not finished for: {missing}")
    print("the check needs all six categories; rerun the Run cell to finish them")

cmp = har.merge(ref, on="category")
cmp["delta"] = cmp.harness - cmp.main_table
print(cmp.to_string(index=False))
print(f"\nmean |delta| = {cmp.delta.abs().mean():.4f}")
print(f"max  |delta| = {cmp.delta.abs().max():.4f}")

if missing:
    print("\nPARTIAL: no verdict until all six categories are in.")
elif cmp.delta.abs().mean() > 0.005:
    print("\nWARNING: deviation above the measured run-to-run noise.")
    print("Before reading the sweeps, set BATCH_TEST = 1 and rerun the giant arm.")
else:
    print("\nwithin noise; the sweeps can be read.")